In [ ]:
# Colab installation
!pip install -q git+https://github.com/2forts/qcirclab_repo.git


In [ ]:
import numpy as np
from itertools import product

from qcirclab import Circuit, cuccaro_adder, controlled_increment


## Utility functions

The helpers below make it easier to inspect states, build small reference permutation unitaries, and verify measurement-free circuits algebraically.


In [ ]:
def pretty_state(state, n_qubits=None, atol=1e-10):
    """Print nonzero amplitudes in computational-basis notation."""
    state = np.asarray(state, dtype=complex).reshape(-1)
    if n_qubits is None:
        n_qubits = int(np.log2(state.size))
    terms = []
    for i, amp in enumerate(state):
        if abs(amp) > atol:
            terms.append(f"({amp:.3g})|{i:0{n_qubits}b}>")
    return " + ".join(terms) if terms else "0"


def circuit_unitary(qc: Circuit) -> np.ndarray:
    """Compute the full unitary matrix of a measurement-free circuit."""
    n = qc.n_qubits
    U = np.zeros((2**n, 2**n), dtype=complex)
    for j in range(2**n):
        basis = np.zeros(2**n, dtype=complex)
        basis[j] = 1.0
        tmp = qc.copy().set_statevector(basis)
        U[:, j] = tmp.statevector()
    return U


def equal_up_to_global_phase(U: np.ndarray, V: np.ndarray, atol: float = 1e-9) -> bool:
    u = U.reshape(-1)
    v = V.reshape(-1)
    idx = None
    for k in range(len(v)):
        if abs(v[k]) > atol and abs(u[k]) > atol:
            idx = k
            break
    if idx is None:
        return np.allclose(U, V, atol=atol)
    phase = u[idx] / v[idx]
    return np.allclose(U, phase * V, atol=atol)


def permutation_unitary(n_qubits, mapping):
    """Build a unitary from a reversible basis-state mapping."""
    dim = 2**n_qubits
    U = np.zeros((dim, dim), dtype=complex)
    seen = set()
    for j in range(dim):
        k = mapping(j)
        if not (0 <= k < dim):
            raise ValueError("mapping produced an invalid basis index")
        if k in seen:
            raise ValueError("mapping is not reversible/permutational")
        seen.add(k)
        U[k, j] = 1.0
    return U


def bits_of(index, n):
    """Return basis bits in book/qcirclab order: q0 is the leftmost bit."""
    return [(index >> (n - 1 - q)) & 1 for q in range(n)]


def index_from_bits(bits):
    out = 0
    for b in bits:
        out = (out << 1) | int(b)
    return out


def bits_to_int_lsb_first(bits):
    return sum(int(b) << i for i, b in enumerate(bits))


def int_to_bits_lsb_first(value, n):
    return [(value >> i) & 1 for i in range(n)]


def set_register_int(bit_list, qubits, value, *, lsb_first=True):
    vals = int_to_bits_lsb_first(value, len(qubits)) if lsb_first else list(map(int, format(value, f"0{len(qubits)}b")))
    for q, b in zip(qubits, vals):
        bit_list[q] = b


# Subsection 4.2.1 **Principles of reversibility and information preservation**

The Toffoli gate computes a reversible AND into a target qubit. Applying it a second time uncomputes the temporary value coherently.


In [ ]:
qc = Circuit(3, name="reversible-demo")

qc.ccx(0, 1, 2)  # compute
qc.ccx(0, 1, 2)  # uncompute

print(qc.draw())
print("Equivalent to identity:", np.allclose(circuit_unitary(qc), np.eye(8)))


# Subsection 4.2.2 **Multi-controlled gates**

Multi-controlled gates generalize the Toffoli gate. They are useful for reversible logic, comparisons, arithmetic conditions, and oracle construction.


In [ ]:
qc = Circuit(7, name="multi_controlled_x")
qc.mcx([0, 1, 2, 3, 4, 5], 6)

print(qc.draw())

state_in = "1111110"
qc_test = Circuit(7).initialize_basis(state_in)
qc_test.compose(qc)
print("Input: ", state_in)
print("Output:", pretty_state(qc_test.statevector()))


In [ ]:
qc = Circuit(11, name="nielsen_chuang_mcx")

controls = [0, 1, 2, 3, 4, 5]
target = 6
ancillas = [7, 8, 9, 10]   # m - 2 ancillas

# Compute: save partial ANDs
qc.ccx(controls[0], controls[1], ancillas[0])   # a0 = c0 AND c1
qc.ccx(ancillas[0], controls[2], ancillas[1])   # a1 = c0 c1 c2
qc.ccx(ancillas[1], controls[3], ancillas[2])   # a2 = c0 c1 c2 c3
qc.ccx(ancillas[2], controls[4], ancillas[3])   # a3 = c0 c1 c2 c3 c4

# Apply controlled-X using the last control and the accumulated AND
qc.ccx(ancillas[3], controls[5], target)

# Uncompute: restore ancillas to |0>
qc.ccx(ancillas[2], controls[4], ancillas[3])
qc.ccx(ancillas[1], controls[3], ancillas[2])
qc.ccx(ancillas[0], controls[2], ancillas[1])
qc.ccx(controls[0], controls[1], ancillas[0])

print(qc.draw())

state_in = "1111110" + "0000"   # controls, target, ancillas
qc_test = Circuit(11).initialize_basis(state_in)
qc_test.compose(qc)

print("Input: ", state_in)
print("Output:", pretty_state(qc_test.statevector()))

# Subsection 4.2.3 **Conventions: endianness and register layout**

In this notebook, `qcirclab` follows the book convention: qubit 0 is the top wire and the leftmost bit in basis labels.


In [ ]:
qc = Circuit(3)
qc.x(0)
qc.x(1)

print("State in book/qcirclab ordering:")
print(pretty_state(qc.statevector()))
print("Qubit labels: q0 q1 q2")


# Subsection 4.3.4 **Constructing and verifying an n-bit adder**

The repository includes a Cuccaro ripple-carry adder. The wire layout is `a[0..n-1], b[0..n-1], carry`, with each arithmetic register interpreted LSB-first.


In [ ]:
def prepare_adder_input(n, a_val, b_val):
    qc = Circuit(2*n + 2)

    cin = 0
    a = list(range(1, 1+n))
    b = list(range(1+n, 1+2*n))
    cout = 1 + 2*n

    bits = [0] * (2*n + 2)
    bits[cin] = 0
    set_register_int(bits, a, a_val, lsb_first=True)
    set_register_int(bits, b, b_val, lsb_first=True)
    bits[cout] = 0

    return qc.initialize_basis("".join(str(x) for x in bits))


def read_adder_output(bitstring, n):
    bits = list(map(int, bitstring))

    cin = bits[0]
    a_bits = bits[1:1+n]
    b_bits = bits[1+n:1+2*n]
    cout = bits[1+2*n]

    return {
        "cin": cin,
        "a": bits_to_int_lsb_first(a_bits),
        "b": bits_to_int_lsb_first(b_bits),
        "carry_out": cout,
    }

n = 3
a_val = 3
b_val = 5

adder = cuccaro_adder(n)
qc = prepare_adder_input(n, a_val, b_val)
qc.compose(adder)

state = qc.statevector()

out = next(
    format(i, f"0{2*n+2}b")
    for i, amp in enumerate(state)
    if abs(amp) > 1e-10
)

print("Adder circuit:")
print(adder.draw())

print("\nOutput state:", pretty_state(state))
print("Bitstring:", out)
print("Decoded output:", read_adder_output(out, n))

print("Expected b:", (a_val + b_val) % (2**n),
      "carry:", (a_val + b_val) >> n)


# Subsection 4.4.1 **Subtraction via additive inverses and modular wrap-around**

For a compact simulation example, modular subtraction is implemented as a reversible permutation on computational-basis states. This is an exact reference model for testing arithmetic behaviour, not an elementary gate decomposition.


In [ ]:
def subtractor_permutation(n):
    # Layout: a[n], b[n]. Map b <- (a - b) mod 2^n, preserving a.
    n_qubits = 2*n
    a = list(range(n))
    b = list(range(n, 2*n))

    def mapping(index):
        bits = bits_of(index, n_qubits)
        a_val = bits_to_int_lsb_first([bits[q] for q in a])
        b_val = bits_to_int_lsb_first([bits[q] for q in b])
        new_b = (a_val - b_val) % (2**n)
        out_bits = bits[:]
        set_register_int(out_bits, b, new_b, lsb_first=True)
        return index_from_bits(out_bits)

    return permutation_unitary(n_qubits, mapping)

n = 3
a_val, b_val = 6, 2
bits = [0] * (2*n)
set_register_int(bits, range(n), a_val)
set_register_int(bits, range(n, 2*n), b_val)

qc = Circuit(2*n).initialize_basis("".join(map(str, bits)))
qc.unitary(subtractor_permutation(n), list(range(2*n)), name="sub")

print("Input a,b =", a_val, b_val)
print("Output:", pretty_state(qc.statevector()))
print("Expected b <- a-b mod 2^n =", (a_val - b_val) % (2**n))


# Subsection 4.4.2 **Comparator circuits**

An equality comparator can be built reversibly by computing bitwise XORs into one register, flipping a flag when all XOR values are zero, and then uncomputing the XORs.


In [ ]:
def equality_comparator(n):
    # Layout: a[n], b[n], flag. The flag flips iff a == b.
    qc = Circuit(2*n + 1, name=f"eq_{n}")
    a = list(range(n))
    b = list(range(n, 2*n))
    flag = 2*n

    for j in range(n):
        qc.cx(a[j], b[j])

    for j in range(n):
        qc.x(b[j])
    qc.mcx(b, flag)
    for j in range(n):
        qc.x(b[j])

    for j in reversed(range(n)):
        qc.cx(a[j], b[j])
    return qc


def test_equality(n, a_val, b_val):
    bits = [0] * (2*n + 1)
    set_register_int(bits, range(n), a_val)
    set_register_int(bits, range(n, 2*n), b_val)
    qc = Circuit(2*n + 1).initialize_basis("".join(map(str, bits)))
    qc.compose(equality_comparator(n))
    return pretty_state(qc.statevector())

n = 3
print(equality_comparator(n).draw())
print("a=5, b=5 ->", test_equality(n, 5, 5))
print("a=5, b=3 ->", test_equality(n, 5, 3))


# Subsection 4.4.3 **Conditional increment/ decrement and controlled adders**


In [ ]:
n = 3
cinc = controlled_increment(n)
print(cinc.draw())

ctrl = 1
x_val = 3
bits = [0] * (n + 1)
bits[0] = ctrl
set_register_int(bits, range(1, n+1), x_val)

qc = Circuit(n + 1).initialize_basis("".join(map(str, bits)))
qc.compose(cinc)

print("Input ctrl, x =", ctrl, x_val)
print("Output:", pretty_state(qc.statevector()))
print("Expected x:", (x_val + ctrl) % (2**n))


# Subsection 4.5.3 **Example: reversible multiplication through a permutation unitary**

A full elementary-gate multiplier is substantially longer than the adders above. For testing and pedagogy, it is useful to start from a reversible reference implementation: preserve the inputs and write the product into a clean product register.


In [ ]:
def multiplier_permutation(n):
    # Layout: a[n], b[n], p[2n]. Map p <- p XOR (a*b), preserving a,b.
    n_qubits = 4*n
    a = list(range(n))
    b = list(range(n, 2*n))
    p = list(range(2*n, 4*n))

    def mapping(index):
        bits = bits_of(index, n_qubits)
        a_val = bits_to_int_lsb_first([bits[q] for q in a])
        b_val = bits_to_int_lsb_first([bits[q] for q in b])
        p_val = bits_to_int_lsb_first([bits[q] for q in p])
        new_p = p_val ^ (a_val * b_val)
        out_bits = bits[:]
        set_register_int(out_bits, p, new_p, lsb_first=True)
        return index_from_bits(out_bits)

    return permutation_unitary(n_qubits, mapping)

n = 2
a_val, b_val = 2, 3
bits = [0] * (4*n)
set_register_int(bits, range(n), a_val)
set_register_int(bits, range(n, 2*n), b_val)
set_register_int(bits, range(2*n, 4*n), 0)

qc = Circuit(4*n).initialize_basis("".join(map(str, bits)))
qc.unitary(multiplier_permutation(n), list(range(4*n)), name="mul")

print("Input a,b =", a_val, b_val)
print("Output:", pretty_state(qc.statevector()))
print("Expected product:", a_val * b_val)


# Subsection 4.6.1 **Modular addition and reduction techniques**

The next reference model computes $b \leftarrow b+a \pmod N$. It is expressed as a permutation unitary, making it exact and reversible on the intended computational-basis subspace.


In [ ]:
def modular_add_permutation(n, modulus):
    # Layout: a[n], b[n]. On valid states a,b < modulus:
    # |a>|b> -> |a>|(a+b) mod modulus>.
    # Invalid states are left unchanged to keep the map reversible.
    n_qubits = 2*n
    a = list(range(n))
    b = list(range(n, 2*n))

    def mapping(index):
        bits = bits_of(index, n_qubits)
        a_val = bits_to_int_lsb_first([bits[q] for q in a])
        b_val = bits_to_int_lsb_first([bits[q] for q in b])

        out_bits = bits[:]

        if a_val < modulus and b_val < modulus:
            new_b = (a_val + b_val) % modulus
            set_register_int(out_bits, b, new_b)

        return index_from_bits(out_bits)

    return permutation_unitary(n_qubits, mapping)

n = 3
N = 5
a_val, b_val = 3, 4
bits = [0] * (2*n)
set_register_int(bits, range(n), a_val)
set_register_int(bits, range(n, 2*n), b_val)

qc = Circuit(2*n).initialize_basis("".join(map(str, bits)))
qc.unitary(modular_add_permutation(n, N), list(range(2*n)), name=f"add_mod_{N}")

print("Input a,b,N =", a_val, b_val, N)
print("Output:", pretty_state(qc.statevector()))
print("Expected b:", (a_val + b_val) % N)


# Subsection 4.6.2 **Controlled modular multiplication**

Controlled modular multiplication is a key block in period-finding and Shor-type circuits.


In [ ]:
def controlled_modmul_permutation(n, a_const, modulus):
    # Layout: ctrl, x[n]. If ctrl=1 and x<modulus: x <- a_const*x mod modulus.
    n_qubits = n + 1
    ctrl = 0
    xreg = list(range(1, n+1))

    if np.gcd(a_const, modulus) != 1:
        raise ValueError("a_const must be coprime to modulus for reversibility")

    def mapping(index):
        bits = bits_of(index, n_qubits)
        x_val = bits_to_int_lsb_first([bits[q] for q in xreg])
        if bits[ctrl] == 1 and x_val < modulus:
            new_x = (a_const * x_val) % modulus
        else:
            new_x = x_val
        out_bits = bits[:]
        set_register_int(out_bits, xreg, new_x)
        return index_from_bits(out_bits)

    return permutation_unitary(n_qubits, mapping)

n = 3
N = 7
a_const = 3
ctrl = 1
x_val = 5

bits = [0] * (n + 1)
bits[0] = ctrl
set_register_int(bits, range(1, n+1), x_val)

qc = Circuit(n + 1).initialize_basis("".join(map(str, bits)))
qc.unitary(controlled_modmul_permutation(n, a_const, N), list(range(n+1)), name="cmm")

print("Input ctrl,x,a,N =", ctrl, x_val, a_const, N)
print("Output:", pretty_state(qc.statevector()))
print("Expected x:", (a_const * x_val) % N)


# Subsection 4.6.3 **Toward modular exponentiation: block architecture**

Modular exponentiation is built from controlled modular multiplications by powers of the base. The example below shows the classical control pattern that becomes a sequence of controlled reversible blocks in a quantum circuit.


In [ ]:
def modular_exponentiation_reference(base, exponent, modulus):
    result = 1
    power = base % modulus
    e = exponent
    while e:
        if e & 1:
            result = (result * power) % modulus
        power = (power * power) % modulus
        e >>= 1
    return result

base = 2
modulus = 15
for exponent in range(8):
    print(f"{base}^{exponent} mod {modulus} =", modular_exponentiation_reference(base, exponent, modulus))


# Subsection 4.7.1 **Truth-table sampling vs. statevector checks**


In [ ]:
def run_adder_once(n, a_val, b_val):
    qc = prepare_adder_input(n, a_val, b_val)
    qc.compose(cuccaro_adder(n))
    state = qc.statevector()
    out = next(
        format(i, f"0{2*n+2}b")
        for i, amp in enumerate(state)
        if abs(amp) > 1e-10
    )
    return read_adder_output(out, n)


n = 3
ok = True

for a_val in range(2**n):
    for b_val in range(2**n):
        out = run_adder_once(n, a_val, b_val)
        expected_b = (a_val + b_val) % (2**n)
        expected_carry = (a_val + b_val) >> n

        if (
            out["cin"] != 0
            or out["a"] != a_val
            or out["b"] != expected_b
            or out["carry_out"] != expected_carry
        ):
            ok = False
            print("Mismatch:", a_val, b_val, out)
            break
    if not ok:
        break

print("All Cuccaro adder basis-state tests passed:", ok)


In [ ]:
n = 2
U = circuit_unitary(equality_comparator(n))
print("Comparator unitary is unitary:", np.allclose(U.conj().T @ U, np.eye(U.shape[0])))
